# S20 · The cheapest thali that still meets nutrition

Our two-food lunch cost ₹31, but it gives only 5.5 mg of iron and the warden wants at least **8 mg**. Two more foods are on offer: curd and palak sabzi. We write the four-food plate as a linear program, let the free CBC solver find the cheapest one, check it by hand, and then change the rules the way a real manager would.


**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on each
  cell, top to bottom, and read the plain-English note above each one.
- New to "the best plan under limits"? Open the primer
  `primers/what_is_optimization.md` for a ten-minute, picture-first version.
- Already confident with code or with linear programming? Look for the cells marked
  **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. It installs PuLP, the small library we
use to write the problem. On your **own machine** you already installed everything
with `uv`, so it does nothing there.

We ask for a PuLP version below 4. PuLP 4.0 (2026) changed how variables are created
and stopped shipping the free CBC solver, so this notebook, and almost every PuLP
tutorial you will find online, needs version 3.

In [1]:
# PuLP is the one library here that Colab does not already include.
import sys
if "google.colab" in sys.modules:
    !pip install -q "pulp<4"
else:
    print("Not on Colab - assuming the libraries are already installed.")

Not on Colab - assuming the libraries are already installed.


In [2]:
import pulp        # a small, readable library for writing linear programs

## Before any code: the checklist

Answer these on paper first. It is the whole skill; the solver is the easy part.

1. **Decision variables**: what do we choose?
2. **Objective**: which one number, and do we push it up or down?
3. **Constraints**: which rules, and do they say "at least" or "at most"?
4. **Non-negativity**: which choices can never be negative?

<details><summary>Our answers (open after you have tried)</summary>

1. Katoris of rice, dal, curd and palak: four variables.
2. Minimise the total cost of the plate.
3. Protein ≥ 30 g, calories ≥ 900, iron ≥ 8 mg: three "at least" rules.
4. All four katoris ≥ 0.
</details>

## Step 1 — the foods and the targets

Per katori. Illustrative numbers, close to real ones, not medical advice. We keep each
column of the table in a dictionary: `price["dal"]` is the price of one katori of dal.

In [3]:
foods = ["rice", "dal", "curd", "palak"]

price    = {"rice": 5,   "dal": 8,   "curd": 10,  "palak": 8}     # rupees
protein  = {"rice": 4,   "dal": 9,   "curd": 6,   "palak": 3}     # grams
calories = {"rice": 200, "dal": 150, "curd": 100, "palak": 50}    # kcal
iron     = {"rice": 0.5, "dal": 2,   "curd": 0.1, "palak": 4}     # milligrams

need_protein, need_calories, need_iron = 30, 900, 8

for f in foods:
    print(f"{f:6s}  Rs {price[f]:3d}  protein {protein[f]:2d} g  "
          f"calories {calories[f]:4d}  iron {iron[f]:4.1f} mg")

rice    Rs   5  protein  4 g  calories  200  iron  0.5 mg
dal     Rs   8  protein  9 g  calories  150  iron  2.0 mg
curd    Rs  10  protein  6 g  calories  100  iron  0.1 mg
palak   Rs   8  protein  3 g  calories   50  iron  4.0 mg


## Step 2 — create the problem and the variables

We want the cheapest plate, so the problem is a **minimise** problem. Then one variable
per food: how many katoris. `lowBound=0` is the non-negativity rule, written straight
into the variable.

In [4]:
thali = pulp.LpProblem("full_thali", pulp.LpMinimize)

# k["dal"] is "how many katoris of dal". One variable per food.
k = {}
for f in foods:
    k[f] = pulp.LpVariable(f, lowBound=0)

print("variables:", list(k.values()))

variables: [rice, dal, curd, palak]


## Step 3 — the objective: total cost

The total cost is the sum over every food of price × katoris. `pulp.lpSum(...)` means
"add all of these up". Adding an expression with no `>=` or `<=` tells PuLP "this is
the number to minimise".

In [5]:
thali += pulp.lpSum(price[f] * k[f] for f in foods)
print("objective:", thali.objective)

objective: 10*curd + 8*dal + 8*palak + 5*rice


## Step 4 — the constraints: one per nutrient

Each nutrient rule is the same sentence with a different column of the table. We give
each rule a name, so we can look it up later.

In [6]:
thali += pulp.lpSum(protein[f] * k[f] for f in foods) >= need_protein, "enough_protein"
thali += pulp.lpSum(calories[f] * k[f] for f in foods) >= need_calories, "enough_calories"
thali += pulp.lpSum(iron[f] * k[f] for f in foods) >= need_iron, "enough_iron"

for name, rule in thali.constraints.items():
    print(name, ":", rule)

enough_protein : 6*curd + 9*dal + 3*palak + 4*rice >= 30.0
enough_calories : 100*curd + 150*dal + 50*palak + 200*rice >= 900.0
enough_iron : 0.1*curd + 2*dal + 4*palak + 0.5*rice >= 8.0


## Step 5 — solve, and read the verdict first

CBC is a free solver that ships inside PuLP 3. We only trust the numbers if the status
says `Optimal`.

In [7]:
thali.solve(pulp.PULP_CBC_CMD(msg=0))
print("status:", pulp.LpStatus[thali.status])

status: Optimal


## Step 6 — read the cheapest plate

`varValue` is the amount the solver chose for a variable; `pulp.value(thali.objective)`
is the cost of the whole plate.

In [8]:
for f in foods:
    print(f"  {f:6s} {k[f].varValue:5.2f} katoris")
print("cost per plate: Rs", pulp.value(thali.objective))

  rice    3.00 katoris
  dal     1.75 katoris
  curd    0.00 katoris
  palak   0.75 katoris
cost per plate: Rs 35.0


Rice stays at 3 katoris. The solver swaps a quarter katori of dal for palak, which is
rich in iron, and skips curd completely: dal gives more protein per rupee. Meeting the
iron target costs ₹4 a plate more than the two-food lunch.

## Step 7 — never trust a solver blindly: check by hand

Add up what the chosen plate really delivers and compare with each target.

In [9]:
got_protein = sum(protein[f] * k[f].varValue for f in foods)
got_calories = sum(calories[f] * k[f].varValue for f in foods)
got_iron = sum(iron[f] * k[f].varValue for f in foods)

print("protein :", round(got_protein, 2), "g   (need", need_protein, ")")
print("calories:", round(got_calories, 2), "     (need", need_calories, ")")
print("iron    :", round(got_iron, 2), "mg   (need", need_iron, ")")

protein : 30.0 g   (need 30 )
calories: 900.0      (need 900 )
iron    : 8.0 mg   (need 8 )


All three are met **exactly**. That is what a corner looks like with more than two
foods: every rule that pins the answer is tight.

## Step 8 — the warden wants curd on the plate

The cheapest plate is rarely the plate people want. The model only knows the rules you
give it. Add a rule: at least half a katori of curd. Then solve again.

In [10]:
thali += k["curd"] >= 0.5, "some_curd"

thali.solve(pulp.PULP_CBC_CMD(msg=0))
print("status:", pulp.LpStatus[thali.status])
for f in foods:
    print(f"  {f:6s} {k[f].varValue:5.2f} katoris")
print("cost per plate: Rs", round(pulp.value(thali.objective), 2))

status: Optimal
  rice    3.00 katoris
  dal     1.35 katoris
  curd    0.50 katoris
  palak   0.94 katoris
cost per plate: Rs 38.32


## Step 9 — a budget the plate cannot meet

Now the accounts office says: at most ₹30 a plate. The cheapest plate that meets every
rule already costs more than that, so no plate can satisfy everything. Watch the
verdict.

In [11]:
thali += pulp.lpSum(price[f] * k[f] for f in foods) <= 30, "budget"

thali.solve(pulp.PULP_CBC_CMD(msg=0))
print("status:", pulp.LpStatus[thali.status])

status: Infeasible


`Infeasible`: the rules contradict each other. That is not a crash; it is true news
about your assumptions. After an Infeasible solve the variables still hold some numbers,
and they mean nothing. Always read the status before the plate.

The fix is a decision for people, not for the solver: raise the budget, relax a
nutrient target, or find cheaper food.

### Stretch (optional) — what is one more gram of protein worth?

Skip this if today is your first taste of the topic. When CBC solves a linear program it
also reports a **shadow price** for each rule: how much the cheapest cost would rise if
that target went up by one unit. We rebuild the plain four-food model (no curd rule, no
budget) and print them.

In [12]:
plain = pulp.LpProblem("plain_thali", pulp.LpMinimize)
k2 = {f: pulp.LpVariable(f, lowBound=0) for f in foods}
plain += pulp.lpSum(price[f] * k2[f] for f in foods)
plain += pulp.lpSum(protein[f] * k2[f] for f in foods) >= need_protein, "enough_protein"
plain += pulp.lpSum(calories[f] * k2[f] for f in foods) >= need_calories, "enough_calories"
plain += pulp.lpSum(iron[f] * k2[f] for f in foods) >= need_iron, "enough_iron"
plain.solve(pulp.PULP_CBC_CMD(msg=0))

print("cost:", pulp.value(plain.objective))
for name, rule in plain.constraints.items():
    print(f"  {name:16s} shadow price Rs {rule.pi:.3f} per extra unit")

cost: 35.0
  enough_protein   shadow price Rs 0.275 per extra unit
  enough_calories  shadow price Rs 0.015 per extra unit
  enough_iron      shadow price Rs 1.600 per extra unit


Read the iron line as a manager would: "each extra milligram of iron the warden asks
for costs about that many rupees per plate". Multiply by 400 lunches a day and the
warden's request has a price tag. A rule with shadow price 0 is not the bottleneck.

## What you just did

You wrote a four-food linear program in the five steps (problem, variables, objective,
rules, solve), read the verdict before the numbers, checked the plate by hand, and then
changed the rules like a real manager: a curd rule made the plate dearer, and a budget
made it impossible. Next notebook: `03_moving_stock.ipynb`, the same recipe for trucks
instead of food.